# DSwinIR → TFLite FP32 256×256
Upload source bundle chứa model.py, natten_pure.py, validate_model.py, tools/restore_tiled.py, convert/prepare_tf_weights.py và convert/direct_tf.py. Dùng checkpoint có provenance rõ. Rain100L single-task không thay thế five-task. Chưa có kết quả chạy Colab được lưu trong notebook này.


In [ ]:
%pip install "numpy==1.26.4" "tensorflow==2.19.1" "torch==2.7.1" gdown


Restart runtime nếu pip yêu cầu, sau đó upload source bundle. Tạo bundle ở máy: `python3 tools/package_conversion.py`. Chỉ giải nén ZIP do bạn tự tạo.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
files.upload()  # chọn dswinir-conversion.zip
with zipfile.ZipFile("dswinir-conversion.zip") as z:
    root=Path("/content/dswinir").resolve()
    for name in z.namelist():
        if not (root/name).resolve().is_relative_to(root):
            raise ValueError("Invalid ZIP path")
    z.extractall(root)
%cd /content/dswinir


In [ ]:
from pathlib import Path
Path("weights").mkdir(exist_ok=True)
import gdown
gdown.download(id="1yQcnuRUepg9NW5DVh3MmBFlA-H1r7y7E", output="weights/rain100l_official.pth", quiet=False)


Bước chuẩn bị chọn CUDA nếu runtime có GPU, còn lại dùng CPU; tắt TF32 để giữ phép tính FP32. Chuyển đổi vẫn cần RAM đáng kể. Không giảm tile về128 để nộp thay256. GPU không thay thế checkpoint five-task hoặc kiểm chứng NATTEN gốc.


In [ ]:
!python convert/prepare_tf_weights.py --checkpoint weights/rain100l_official.pth --out convert/prepared256 --size 256 --device auto


In [ ]:
!TF_CPP_MIN_LOG_LEVEL=1 python convert/direct_tf.py --prepared convert/prepared256


In [ ]:
from google.colab import files
from pathlib import Path
p=Path("convert/prepared256/dswinir_256_fp32.tflite")
assert p.is_file(), "Conversion/parity chưa qua; đọc log, không dùng artifact cũ"
files.download(str(p))
files.download("convert/prepared256/conversion-report.json")
